In [5]:
import os
import logging
from datetime import datetime

import pandas as pd

In [7]:
# read pkl to get id
ID_old = pd.read_pickle("../results/result_df_log.pkl")["ID"].to_list()[-1]
ID_new = ID_old + 1

In [8]:
# Run params
TS      = datetime.now().strftime("%Y%m%d_%H%M%S")
SDG     = "none" # name of used generation model 
RUN     = "test" # status of run test or final run 
DATASET = "yeast" # which dataset was used

In [9]:
# set up logging
logger = logging.getLogger(__name__)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
formatter = logging.Formatter("%(levelname)s: %(message)s")
logger.setLevel(logging.INFO)

# log to stdout
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# log to file
log_file = f"../logs/{ID_new}_{TS}_{RUN}_{SDG}_{DATASET}_log.txt"
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

In [ ]:
logger.info("-"*50)
logger.info(f"Timestamp: {TS}")
logger.info(f"Dataset: {DATASET}")
logger.info(f"Synthetic Data Generation Algo: {SDG}")
logger.info(f"Type of Run: {RUN}")

INFO: --------------------------------------------------
INFO: Timestamp: 20241118_110843
INFO: Dataset: yeast
INFO: Synthetic Data Generation Algo: none
INFO: Classification Model: RF
INFO: Type of Run: test


In [12]:
df = pd.read_csv("../data/raw/yeast.csv")

In [13]:
data = df[(df["localization_site"] == "CYT") | (df["localization_site"] == "ME3")]

In [14]:
class_maj_grp = data["localization_site"].value_counts().values[0]
class_min_grp = data["localization_site"].value_counts().values[1]

In [15]:
logger.info("-"*50)
logger.info("Dataset infos")
logger.info(f"Class imbalance ratio: {class_min_grp * 100 / (class_maj_grp + class_min_grp)}")
logger.info(f"# Majority class: {class_maj_grp}")
logger.info(f"# Minority class: {class_min_grp}")

INFO: --------------------------------------------------
INFO: Dataset infos
INFO: Class imbalance ratio: 26.038338658146966
INFO: # Majority class: 463
INFO: # Minority class: 163


In [27]:
# pre process
# certain classification models need numbers as the y variable, like XGBoost
# 1 for minority class and 0 for majority class
data["localization_site"] = data["localization_site"].map({"CYT":0, "ME3": 1})


/var/folders/gv/d21k9b7d7v9d382g4l3v4rqm0000gn/T/ipykernel_5335/415948857.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["localization_site"] = data["localization_site"].map({"CYT":0, "ME3": 1})


In [28]:
data

,mcg,gvh,alm,mit,erl,pox,vac,nuc,localization_site
5,0.51,0.40,0.56,0.17,0.5,0.5,0.49,0.22,0
9,0.40,0.39,0.60,0.15,0.5,0.0,0.58,0.30,0
12,0.40,0.42,0.57,0.35,0.5,0.0,0.53,0.25,0
15,0.46,0.44,0.52,0.11,0.5,0.0,0.50,0.22,0
16,0.47,0.39,0.50,0.11,0.5,0.0,0.49,0.40,0
...,...,...,...,...,...,...,...,...,...
1475,0.71,0.50,0.50,0.18,0.5,0.0,0.46,0.22,0
1476,0.61,0.48,0.54,0.25,0.5,0.0,0.50,0.22,0
1477,0.38,0.32,0.64,0.41,0.5,0.0,0.44,0.11,0
1478,0.38,0.40,0.66,0.35,0.5,0.0,0.43,0.11,0


In [59]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.svm import SVC

from xgboost import XGBClassifier
import lightgbm as lgb

import numpy as np

In [29]:
X = data.iloc[:, :-1]
y = data["localization_site"]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
logger.info("-"*50)
logger.info("Shape of train and test data")
logger.info(f"Timestamp: {datetime.now().strftime('%Y%m%d_%H%M%S')}")
logger.info(f"X_train: {X_train.shape}")
logger.info(f"y_train: {y_train.shape}")
logger.info(f"X_test: {X_test.shape}")
logger.info(f"y_test: {y_test.shape}")

INFO: --------------------------------------------------
INFO: Shape of train and test data
INFO: Timestamp: 20241118_164736
INFO: X_train: (500, 8)
INFO: y_train: (500,)
INFO: X_test: (126, 8)
INFO: y_test: (126,)


In [32]:
logger.info("-"*50)
logger.info(f"Stating training of classifier: {datetime.now().strftime('%Y%m%d_%H%M%S')}")

INFO: --------------------------------------------------
INFO: Stating training of classifier: 20241118_164738


In [ ]:
clfs = { 
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost":  XGBClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42, algorithm="SAMME"),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "KNeighbors": KNeighborsClassifier(),
    "SVM": SVC(random_state=42)
}

In [61]:
logger.info("Classification Models:")
for name, clf in clfs.items():
    logger.info(f"{name}: {clf.get_params()}")
    

INFO: Classification Models:
INFO: Logistic Regression: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
INFO: Random Forest: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
INFO: XGBoost: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None

In [62]:
# save f1_socres in results dict
results = {}

for name, clf in clfs.items():
    logger.info(f"Results for {name}")
    # train model
    clf.fit(X_train, y_train)

    # predcit
    y_pred = clf.predict(X_test)

    # calcualte f1_score
    accuracy_score_value = accuracy_score(y_test, y_pred)
    f1_score_value = float(f1_score(list(y_test), list(y_pred), pos_label=1))
    # log resutls
    
    logger.info(f"Accuracy Score: {accuracy_score_value}")
    logger.info(f"F1-Score of minority class: {f1_score_value}")
    logger.info(f"{classification_report(y_test, y_pred)}")

    # add results to dict
    results[name] = (accuracy_score_value, f1_score_value)

INFO: Results for Logistic Regression
INFO: Accuracy Score: 0.8412698412698413
INFO: F1-Score of minority class: 0.6774193548387096
INFO:               precision    recall  f1-score   support

           0       0.81      1.00      0.89        85
           1       1.00      0.51      0.68        41

    accuracy                           0.84       126
   macro avg       0.90      0.76      0.79       126
weighted avg       0.87      0.84      0.82       126

INFO: Results for Random Forest
INFO: Accuracy Score: 0.9523809523809523
INFO: F1-Score of minority class: 0.926829268292683
INFO:               precision    recall  f1-score   support

           0       0.96      0.96      0.96        85
           1       0.93      0.93      0.93        41

    accuracy                           0.95       126
   macro avg       0.95      0.95      0.95       126
weighted avg       0.95      0.95      0.95       126

INFO: Results for XGBoost
INFO: Accuracy Score: 0.9603174603174603
INFO: F1-S

[LightGBM] [Info] Number of positive: 122, number of negative: 378
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 234
[LightGBM] [Info] Number of data points in the train set: 500, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.244000 -> initscore=-1.130873
[LightGBM] [Info] Start training from score -1.130873
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

INFO: Results for SVM
INFO: Accuracy Score: 0.9285714285714286
INFO: F1-Score of minority class: 0.8860759493670886
INFO:               precision    recall  f1-score   support

           0       0.93      0.96      0.95        85
           1       0.92      0.85      0.89        41

    accuracy                           0.93       126
   macro avg       0.93      0.91      0.92       126
weighted avg       0.93      0.93      0.93       126



In [63]:
results

{'Logistic Regression': (0.8412698412698413, 0.6774193548387096),
 'Random Forest': (0.9523809523809523, 0.926829268292683),
 'XGBoost': (0.9603174603174603, 0.9382716049382716),
 'AdaBoost': (0.9603174603174603, 0.9382716049382716),
 'LightGBM': (0.9603174603174603, 0.9382716049382716),
 'KNeighbors': (0.9206349206349206, 0.8717948717948718),
 'SVM': (0.9285714285714286, 0.8860759493670886)}

In [ ]:
logger.info(f"Finished training of classifier: {datetime.now().strftime('%Y%m%d_%H%M%S')}")

INFO: Finished training of classifier: 20241118_111823


In [98]:
logger.info("-"*50)
logger.info("Saving results...")

INFO: --------------------------------------------------
INFO: Saving results...


In [99]:
import pickle

In [ ]:
# read pkl file
df = pd.read_pickle("../results/result_df_log.pkl")

In [131]:
df

,ID,Timestamp,RUN,Dataset,SDG,CLF,Accuracy,F1-Score
0,1,20241118_110843,test,yeast,none,RF,0.726257,0.71345


In [64]:
results

{'Logistic Regression': (0.8412698412698413, 0.6774193548387096),
 'Random Forest': (0.9523809523809523, 0.926829268292683),
 'XGBoost': (0.9603174603174603, 0.9382716049382716),
 'AdaBoost': (0.9603174603174603, 0.9382716049382716),
 'LightGBM': (0.9603174603174603, 0.9382716049382716),
 'KNeighbors': (0.9206349206349206, 0.8717948717948718),
 'SVM': (0.9285714285714286, 0.8860759493670886)}

In [66]:
pd.DataFrame([
    {
        "ID": 0,
        "Timestamp": TS,
        "RUN": "init",
        "SDG": "none",
        "LR_accuracy": results["Logistic Regression"][0],
        "LR_f1-score": results["Logistic Regression"][1],
        "RF_accuracy": results["Random Forest"][0],
        "RF_f1-score": results["Random Forest"][1],
        "XGB_accuracy": results["XGBoost"][0],
        "XGB_f1-score": results["XGBoost"][1],
        "Ada_accuracy": results["AdaBoost"][0],
        "Ada_f1-score": results["AdaBoost"][1],
        "LGBM_accuracy": results["LightGBM"][0],
        "LGBM_f1-score": results["LightGBM"][1],
        "KNN_accuracy": results["KNeighbors"][0],
        "KNN_f1-score": results["KNeighbors"][1],
        "SVM_accuracy": results["SVM"][0],
        "SVM_f1-score": results["SVM"][1]
    }
])

,ID,Timestamp,RUN,SDG,LR_accuracy,LR_f1-score,RF_accuracy,RF_f1-score,XGB_accuracy,XGB_f1-score,Ada_accuracy,Ada_f1-score,LGBM_accuracy,LGBM_f1-score,KNN_accuracy,KNN_f1-score,SVM_accuracy,SVM_f1-score
0,0,20241118_164050,init,none,0.84127,0.677419,0.952381,0.926829,0.960317,0.938272,0.960317,0.938272,0.960317,0.938272,0.920635,0.871795,0.928571,0.886076


In [135]:
df_save = pd.concat([df, df_new_run])

In [136]:
df_save

,ID,Timestamp,RUN,Dataset,SDG,CLF,Accuracy,F1-Score
0,1,20241118_110843,test,yeast,none,RF,0.726257,0.71345
0,2,20241118_110843,test,yeast,none,RF,0.726257,0.71345


In [ ]:
# save to pickle file 
df_save.to_pickle('../results/result_df_log.pkl')

In [ ]:
df_save.to_csv("../results/result_df_log.csv", index=False)

In [106]:
logger.info("Results saved")

INFO: Results saved
